# CICIoT2023 dataset inspection

Select the existing project environment as the notebook kernel:
`.venv`.

Run the kernel check first, then run the download cell when you are ready. KaggleHub uses its own dataset cache; this notebook does not install packages. The inspection reads CSV headers and scans only the label column in chunks, without loading the full dataset into memory.


In [1]:
import sys
from pathlib import Path
from importlib.metadata import version

repo_root = Path.cwd().resolve()
if repo_root.name == "notebooks":
    repo_root = repo_root.parent
sys.path.insert(0, str(repo_root))

expected_env = (repo_root / ".venv").resolve()
print("Interpreter:", sys.executable)
print("Python:", sys.version.split()[0])
assert Path(sys.prefix) == expected_env, "Select the project's .venv kernel in VS Code."
assert sys.version_info[:2] == (3, 12), "Select the Python 3.12 kernel."

import ipykernel
import pandas as pd
import kagglehub

for package in ("ipykernel", "pandas", "kagglehub"):
    print(f"{package}: {version(package)}")


Interpreter: .venv/bin/python
Python: 3.12.16


ipykernel: 7.4.0
pandas: 3.1.1
kagglehub: 1.0.3


.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Download or reuse the cached dataset

This cell uses `CICIOT2023_DATA_DIR` if set, or downloads or reuses `himadri07/ciciot2023` version 1. If Kaggle requests authentication, configure your Kaggle credentials before retrying.

In [2]:
from src.batch.dataset_paths import dataset_root

dataset_path = dataset_root()
print("Dataset directory:", dataset_path)


Dataset directory: <dataset directory>


## List CSV files and inspect their columns

Run after the download cell. Sizes are reported in bytes and MiB; paths are relative to the dataset directory. Each file's header is inspected separately because schemas may differ. Files that cannot be read remain in the table with an error message.


In [3]:
if "dataset_path" not in globals():
    raise RuntimeError("Run the download cell first to set dataset_path.")

csv_files = sorted(
    path for path in dataset_path.rglob("*")
    if path.is_file() and path.suffix.lower() == ".csv"
)
print(f"Found {len(csv_files):,} CSV files in {dataset_path}")
if not csv_files:
    raise FileNotFoundError(f"No CSV files found under {dataset_path}")

records = []
for csv_path in csv_files:
    size_bytes = csv_path.stat().st_size
    record = {
        "csv": str(csv_path.relative_to(dataset_path)),
        "size_bytes": size_bytes,
        "size_MiB": round(size_bytes / (1024 ** 2), 2),
        "column_count": None,
        "column_names": None,
        "error": None,
    }
    try:
        columns = pd.read_csv(csv_path, nrows=0).columns.tolist()
        record.update(column_count=len(columns), column_names=columns)
    except Exception as exc:
        record["error"] = f"{type(exc).__name__}: {exc}"
    records.append(record)

csv_inventory = pd.DataFrame(records)
print(f"Total CSV size: {csv_inventory['size_bytes'].sum() / (1024 ** 3):.2f} GiB")
with pd.option_context("display.max_rows", None, "display.max_colwidth", None):
    display(csv_inventory)


Found 3 CSV files in <dataset directory>
Total CSV size: 2.16 GiB


,csv,size_bytes,size_MiB,column_count,column_names,error
0,CICIOT23/test/test.csv,347958926,331.84,47,"[flow_duration, Header_Length, Protocol Type, Duration, Rate, Srate, Drate, fin_flag_number, syn_flag_number, rst_flag_number, psh_flag_number, ack_flag_number, ece_flag_number, cwr_flag_number, ack_count, syn_count, fin_count, urg_count, rst_count, HTTP, HTTPS, DNS, Telnet, SMTP, SSH, IRC, TCP, UDP, DHCP, ARP, ICMP, IPv, LLC, Tot sum, Min, Max, AVG, Std, Tot size, IAT, Number, Magnitue, Radius, Covariance, Variance, Weight, label]",None
1,CICIOT23/train/train.csv,1623446296,1548.24,47,"[flow_duration, Header_Length, Protocol Type, Duration, Rate, Srate, Drate, fin_flag_number, syn_flag_number, rst_flag_number, psh_flag_number, ack_flag_number, ece_flag_number, cwr_flag_number, ack_count, syn_count, fin_count, urg_count, rst_count, HTTP, HTTPS, DNS, Telnet, SMTP, SSH, IRC, TCP, UDP, DHCP, ARP, ICMP, IPv, LLC, Tot sum, Min, Max, AVG, Std, Tot size, IAT, Number, Magnitue, Radius, Covariance, Variance, Weight, label]",None
2,CICIOT23/validation/validation.csv,347921865,331.80,47,"[flow_duration, Header_Length, Protocol Type, Duration, Rate, Srate, Drate, fin_flag_number, syn_flag_number, rst_flag_number, psh_flag_number, ack_flag_number, ece_flag_number, cwr_flag_number, ack_count, syn_count, fin_count, urg_count, rst_count, HTTP, HTTPS, DNS, Telnet, SMTP, SSH, IRC, TCP, UDP, DHCP, ARP, ICMP, IPv, LLC, Tot sum, Min, Max, AVG, Std, Tot size, IAT, Number, Magnitue, Radius, Covariance, Variance, Weight, label]",None


## Raw label counts by split

Read only `label`, in chunks of 100,000 rows. Disable pandas' default NA conversion and preserve case, whitespace, and literal strings such as `NA` and `null`. Only an empty label field (`""`) counts as missing; whitespace-only strings remain distinct labels. No binary mapping is applied.

Percentages use **all rows in that split**, including missing labels, as the denominator. The decimal percentage is displayed to eight decimal places; `exact_percentage` gives the exact rational percentage without rounding. `raw_label_repr` uses Python `repr` so leading/trailing whitespace and empty strings are visible. `raw_label` retains the parsed CSV field unchanged.


In [4]:
from collections import Counter
from fractions import Fraction

uploader_counts = {
    "train": 5_491_971,
    "validation": 1_176_851,
    "test": 1_176_851,
}
chunk_size = 100_000
split_summaries = []
label_records = []
raw_label_counts = {}

for split, expected_rows in uploader_counts.items():
    matches = [path for path in csv_files if path.stem == split]
    if len(matches) != 1:
        raise ValueError(f"Expected one {split}.csv, found {len(matches)}: {matches}")
    csv_path = matches[0]
    counts = Counter()
    total_rows = 0
    for chunk in pd.read_csv(
        csv_path,
        usecols=["label"],
        dtype={"label": str},
        keep_default_na=False,
        na_filter=False,
        skip_blank_lines=False,
        chunksize=chunk_size,
    ):
        labels = chunk["label"]
        total_rows += len(labels)
        counts.update(labels.value_counts(dropna=False).to_dict())

    assert sum(counts.values()) == total_rows, f"Label count mismatch for {split}"
    missing_labels = counts.get("", 0)
    raw_label_counts[split] = counts
    split_summaries.append({
        "split": split,
        "total_rows": total_rows,
        "missing_labels": missing_labels,
        "uploader_rows": expected_rows,
        "difference": total_rows - expected_rows,
        "matches_uploader": total_rows == expected_rows,
    })
    for raw_label, count in sorted(counts.items()):
        exact_percentage = Fraction(100 * count, total_rows)
        label_records.append({
            "split": split,
            "raw_label": raw_label,
            "raw_label_repr": repr(raw_label),
            "is_missing": raw_label == "",
            "count": count,
            "percentage": float(exact_percentage),
            "exact_percentage": str(exact_percentage) + "%",
        })
    print(f"{split}: {total_rows:,} rows, {missing_labels:,} missing labels, "
          f"{len(counts):,} distinct raw labels")

split_summary = pd.DataFrame(split_summaries)
label_distribution = pd.DataFrame(label_records)
print("\nComparison with uploader row counts:")
display(split_summary)
for split in uploader_counts:
    print(f"\n{split}: raw label distribution (percentage of all split rows)")
    with pd.option_context(
        "display.max_rows", None,
        "display.max_columns", None,
        "display.max_colwidth", None,
        "display.float_format", "{:.8f}".format,
    ):
        display(label_distribution.loc[label_distribution["split"] == split].reset_index(drop=True))


train: 5,491,971 rows, 0 missing labels, 34 distinct raw labels


validation: 1,176,851 rows, 0 missing labels, 34 distinct raw labels


test: 1,176,851 rows, 0 missing labels, 34 distinct raw labels

Comparison with uploader row counts:


,split,total_rows,missing_labels,uploader_rows,difference,matches_uploader
0,train,5491971,0,5491971,0,True
1,validation,1176851,0,1176851,0,True
2,test,1176851,0,1176851,0,True



train: raw label distribution (percentage of all split rows)


,split,raw_label,raw_label_repr,is_missing,count,percentage,exact_percentage
0,train,Backdoor_Malware,'Backdoor_Malware',False,392,0.00713769,39200/5491971%
1,train,BenignTraffic,'BenignTraffic',False,129538,2.35867961,12953800/5491971%
2,train,BrowserHijacking,'BrowserHijacking',False,665,0.01210859,66500/5491971%
3,train,CommandInjection,'CommandInjection',False,620,0.01128921,62000/5491971%
4,train,DDoS-ACK_Fragmentation,'DDoS-ACK_Fragmentation',False,33581,0.61145625,3358100/5491971%
5,train,DDoS-HTTP_Flood,'DDoS-HTTP_Flood',False,3371,0.06138051,337100/5491971%
6,train,DDoS-ICMP_Flood,'DDoS-ICMP_Flood',False,848088,15.44232480,9423200/610219%
7,train,DDoS-ICMP_Fragmentation,'DDoS-ICMP_Fragmentation',False,53046,0.96588274,589400/610219%
8,train,DDoS-PSHACK_Flood,'DDoS-PSHACK_Flood',False,481254,8.76286492,16041800/1830657%
9,train,DDoS-RSTFINFlood,'DDoS-RSTFINFlood',False,475441,8.65701949,47544100/5491971%



validation: raw label distribution (percentage of all split rows)


,split,raw_label,raw_label_repr,is_missing,count,percentage,exact_percentage
0,validation,Backdoor_Malware,'Backdoor_Malware',False,93,0.00790244,9300/1176851%
1,validation,BenignTraffic,'BenignTraffic',False,27519,2.33835889,2751900/1176851%
2,validation,BrowserHijacking,'BrowserHijacking',False,158,0.01342566,15800/1176851%
3,validation,CommandInjection,'CommandInjection',False,132,0.01121637,13200/1176851%
4,validation,DDoS-ACK_Fragmentation,'DDoS-ACK_Fragmentation',False,7194,0.61129234,719400/1176851%
5,validation,DDoS-HTTP_Flood,'DDoS-HTTP_Flood',False,706,0.05999060,70600/1176851%
6,validation,DDoS-ICMP_Flood,'DDoS-ICMP_Flood',False,182011,15.46593409,18201100/1176851%
7,validation,DDoS-ICMP_Fragmentation,'DDoS-ICMP_Fragmentation',False,11430,0.97123595,1143000/1176851%
8,validation,DDoS-PSHACK_Flood,'DDoS-PSHACK_Flood',False,102985,8.75089540,10298500/1176851%
9,validation,DDoS-RSTFINFlood,'DDoS-RSTFINFlood',False,101563,8.63006447,10156300/1176851%



test: raw label distribution (percentage of all split rows)


,split,raw_label,raw_label_repr,is_missing,count,percentage,exact_percentage
0,test,Backdoor_Malware,'Backdoor_Malware',False,89,0.00756255,8900/1176851%
1,test,BenignTraffic,'BenignTraffic',False,27709,2.35450367,2770900/1176851%
2,test,BrowserHijacking,'BrowserHijacking',False,134,0.01138632,13400/1176851%
3,test,CommandInjection,'CommandInjection',False,119,0.01011173,11900/1176851%
4,test,DDoS-ACK_Fragmentation,'DDoS-ACK_Fragmentation',False,7292,0.61961965,729200/1176851%
5,test,DDoS-HTTP_Flood,'DDoS-HTTP_Flood',False,709,0.06024552,70900/1176851%
6,test,DDoS-ICMP_Flood,'DDoS-ICMP_Flood',False,180447,15.33303706,18044700/1176851%
7,test,DDoS-ICMP_Fragmentation,'DDoS-ICMP_Fragmentation',False,11402,0.96885672,1140200/1176851%
8,test,DDoS-PSHACK_Flood,'DDoS-PSHACK_Flood',False,103326,8.77987103,10332600/1176851%
9,test,DDoS-RSTFINFlood,'DDoS-RSTFINFlood',False,101819,8.65181743,10181900/1176851%


### Inspect the exact benign spelling

The following list shows every distinct raw label using `repr`. Confirm the benign label here before choosing any binary mapping; the notebook leaves all labels unchanged.


In [5]:
all_raw_labels = sorted(set().union(*(counts.keys() for counts in raw_label_counts.values())))
for raw_label in all_raw_labels:
    print(repr(raw_label))


'Backdoor_Malware'
'BenignTraffic'
'BrowserHijacking'
'CommandInjection'
'DDoS-ACK_Fragmentation'
'DDoS-HTTP_Flood'
'DDoS-ICMP_Flood'
'DDoS-ICMP_Fragmentation'
'DDoS-PSHACK_Flood'
'DDoS-RSTFINFlood'
'DDoS-SYN_Flood'
'DDoS-SlowLoris'
'DDoS-SynonymousIP_Flood'
'DDoS-TCP_Flood'
'DDoS-UDP_Flood'
'DDoS-UDP_Fragmentation'
'DNS_Spoofing'
'DictionaryBruteForce'
'DoS-HTTP_Flood'
'DoS-SYN_Flood'
'DoS-TCP_Flood'
'DoS-UDP_Flood'
'MITM-ArpSpoofing'
'Mirai-greeth_flood'
'Mirai-greip_flood'
'Mirai-udpplain'
'Recon-HostDiscovery'
'Recon-OSScan'
'Recon-PingSweep'
'Recon-PortScan'
'SqlInjection'
'Uploading_Attack'
'VulnerabilityScan'
'XSS'


## Exact overlap across splits and duplicates within splits

This check scans **every row** and compares the parsed CSV strings exactly across all 47 columns, including `label`, and separately across the 46 feature columns. It preserves case and whitespace and does not coerce numeric strings or map labels. SQLite stores complete values on disk and compares them directly, so there are no hash collisions to verify. Only a 10,000-row batch and a bounded database page cache are held in memory; the temporary database is removed after the run.

**Distinct shared full rows** counts each complete row present in both files once. **Distinct shared feature patterns** ignores `label` and counts each shared feature combination once. **Rows with features in the other split** counts every occurrence in the named split, including repeated copies, with its percentage calculated against that split's total rows. Thus a repeated test pattern counts once as a distinct pattern but every matching test row contributes to test coverage. **Extra duplicate copies** within a split is total rows minus distinct full rows. A feature match may have different labels; neither overlap metric proves the files are independent samples.

In [7]:
import csv
import json
import sqlite3
import tempfile
import time
from collections import Counter
from itertools import combinations
from pathlib import Path

# Disk-backed exact keys: JSON arrays preserve field boundaries and raw strings.
# No hashes, numeric coercion, whitespace stripping, or label mapping are used.
splits = ("train", "validation", "test")
files = {}
headers = {}
for split in splits:
    matches = [p for p in csv_files if p.stem == split]
    if len(matches) != 1:
        raise ValueError(f"Expected exactly one CSV for {split}: {matches}")
    files[split] = matches[0]
    with files[split].open(newline="", encoding="utf-8") as handle:
        headers[split] = next(csv.reader(handle))
reference = headers["train"]
assert len(reference) == 47 and len(set(reference)) == 47
assert reference.count("label") == 1
assert all(header == reference for header in headers.values()), "CSV schemas/order differ"
label_index = reference.index("label")
feature_indices = [i for i in range(len(reference)) if i != label_index]
assert len(feature_indices) == 46

started = time.monotonic()
with tempfile.TemporaryDirectory(prefix="ciciot-overlap-", dir="/tmp") as temp_dir:
    connection = sqlite3.connect(str(Path(temp_dir) / "exact_rows.sqlite"))
    try:
        connection.execute("PRAGMA journal_mode=OFF")
        connection.execute("PRAGMA synchronous=OFF")
        connection.execute("PRAGMA temp_store=FILE")
        connection.execute("PRAGMA cache_size=-131072")  # About 128 MiB of page cache.
        connection.execute('''CREATE TABLE full_rows (
            features TEXT NOT NULL COLLATE BINARY,
            label TEXT NOT NULL COLLATE BINARY,
            train INTEGER NOT NULL DEFAULT 0,
            validation INTEGER NOT NULL DEFAULT 0,
            test INTEGER NOT NULL DEFAULT 0,
            PRIMARY KEY (features, label)
        ) WITHOUT ROWID''')
        total_rows = {}
        for split in splits:
            sql = f'''INSERT INTO full_rows (features, label, {split}) VALUES (?, ?, ?)
                      ON CONFLICT(features, label) DO UPDATE
                      SET {split} = {split} + excluded.{split}'''
            pending = Counter()
            total = 0
            with files[split].open(newline="", encoding="utf-8") as handle:
                reader = csv.reader(handle, strict=True)
                next(reader)
                for row in reader:
                    if len(row) != len(reference):
                        raise ValueError(f"{split}: malformed record ending on line {reader.line_num}")
                    features = json.dumps([row[i] for i in feature_indices], ensure_ascii=True, separators=(",", ":"))
                    pending[(features, row[label_index])] += 1
                    total += 1
                    if total % 10_000 == 0:
                        connection.executemany(sql, ((f, label, n) for (f, label), n in pending.items()))
                        connection.commit()
                        pending.clear()
                    if total % 500_000 == 0:
                        print(f"{split}: scanned {total:,} rows ({time.monotonic() - started:.0f}s)", flush=True)
            if pending:
                connection.executemany(sql, ((f, label, n) for (f, label), n in pending.items()))
                connection.commit()
            total_rows[split] = total
            print(f"{split}: complete, {total:,} rows", flush=True)

        within_records = []
        for split in splits:
            summed_rows, distinct_rows = connection.execute(
                f"SELECT COALESCE(SUM({split}), 0), COUNT(*) FROM full_rows WHERE {split} > 0"
            ).fetchone()
            assert summed_rows == total_rows[split]
            within_records.append({
                "split": split, "total_rows": summed_rows,
                "distinct_full_rows": distinct_rows,
                "extra_duplicate_copies": summed_rows - distinct_rows,
            })
        print("Aggregating distinct feature combinations on disk...", flush=True)
        connection.execute('''CREATE TABLE feature_patterns (
            features TEXT PRIMARY KEY COLLATE BINARY,
            train INTEGER NOT NULL,
            validation INTEGER NOT NULL,
            test INTEGER NOT NULL
        ) WITHOUT ROWID''')
        connection.execute('''INSERT INTO feature_patterns
            SELECT features, SUM(train), SUM(validation), SUM(test)
            FROM full_rows GROUP BY features''')
        connection.commit()
        pair_records = []
        for a, b in combinations(splits, 2):
            shared_full = connection.execute(
                f"SELECT COUNT(*) FROM full_rows WHERE {a} > 0 AND {b} > 0"
            ).fetchone()[0]
            shared_features, a_covered, b_covered = connection.execute(
                f"SELECT COUNT(*), COALESCE(SUM({a}),0), COALESCE(SUM({b}),0) "
                f"FROM feature_patterns WHERE {a} > 0 AND {b} > 0"
            ).fetchone()
            pair_records.append({
                "split_a": a, "split_b": b,
                "distinct_shared_full_rows": shared_full,
                "distinct_shared_feature_patterns": shared_features,
                "a_rows_with_features_in_b": a_covered,
                "a_percentage": 100 * a_covered / total_rows[a] if total_rows[a] else 0,
                "b_rows_with_features_in_a": b_covered,
                "b_percentage": 100 * b_covered / total_rows[b] if total_rows[b] else 0,
            })
        within_split_duplicates = pd.DataFrame(within_records)
        pairwise_overlap = pd.DataFrame(pair_records)
    finally:
        connection.close()

print(f"Checked every row in {time.monotonic() - started:.1f} seconds. Temporary database removed.")
print("Within-split duplicate counts:")
display(within_split_duplicates)
print("Pairwise overlap (percentages use each respective file's total row count):")
with pd.option_context("display.max_columns", None, "display.width", 220, "display.float_format", "{:.8f}".format):
    display(pairwise_overlap)


print("\nPRINTED ANSWERS")
for record in within_records:
    print(f"{record['split']}: {record['total_rows']:,} total rows; "
          f"{record['distinct_full_rows']:,} distinct full rows; "
          f"{record['extra_duplicate_copies']:,} extra duplicate copies.")
for record in pair_records:
    a, b = record["split_a"], record["split_b"]
    print(f"\n{a} vs {b}:")
    print(f"  Distinct shared full rows (including label): {record['distinct_shared_full_rows']:,}")
    print(f"  Distinct shared feature patterns (excluding label): {record['distinct_shared_feature_patterns']:,}")
    print(f"  {a} rows whose features appear in {b}: "
          f"{record['a_rows_with_features_in_b']:,} / {total_rows[a]:,} "
          f"({record['a_percentage']:.8f}%)")
    print(f"  {b} rows whose features appear in {a}: "
          f"{record['b_rows_with_features_in_a']:,} / {total_rows[b]:,} "
          f"({record['b_percentage']:.8f}%)")
print("\nThe splits overlap." if any(r["distinct_shared_feature_patterns"] for r in pair_records)
      else "\nThe splits have no shared feature patterns.")


train: scanned 500,000 rows (15s)
train: scanned 1,000,000 rows (32s)
train: scanned 1,500,000 rows (51s)
train: scanned 2,000,000 rows (71s)
train: scanned 2,500,000 rows (94s)
train: scanned 3,000,000 rows (117s)
train: scanned 3,500,000 rows (142s)
train: scanned 4,000,000 rows (166s)
train: scanned 4,500,000 rows (190s)
train: scanned 5,000,000 rows (215s)
train: complete, 5,491,971 rows
validation: scanned 500,000 rows (268s)
validation: scanned 1,000,000 rows (297s)
validation: complete, 1,176,851 rows
test: scanned 500,000 rows (338s)
test: scanned 1,000,000 rows (368s)
test: complete, 1,176,851 rows
Aggregating distinct feature combinations on disk...
Checked every row in 463.3 seconds. Temporary database removed.
Within-split duplicate counts:


split,total_rows,distinct_full_rows,extra_duplicate_copies
train,5491971,5357406,134565
validation,1176851,1170554,6297
test,1176851,1170467,6384


     split  total_rows  distinct_full_rows  extra_duplicate_copies
     train     5491971             5357406                  134565
validation     1176851             1170554                    6297
      test     1176851             1170467                    6384
Pairwise overlap (percentages use each respective file's total row count):


split_a,split_b,distinct_shared_full_rows,distinct_shared_feature_patterns,a_rows_with_features_in_b,a_percentage,b_rows_with_features_in_a,b_percentage
train,validation,57744,57744,57744,1.05142580,57744,4.90665343
train,test,57847,57847,57847,1.05330126,57847,4.91540560
validation,test,12422,12422,12422,1.05552869,12422,1.05552869


   split_a    split_b  distinct_shared_full_rows  distinct_shared_feature_patterns  a_rows_with_features_in_b  a_percentage  b_rows_with_features_in_a  b_percentage
     train validation                      57744                             57744                      57744    1.05142580                      57744    4.90665343
     train       test                      57847                             57847                      57847    1.05330126                      57847    4.91540560
validation       test                      12422                             12422                      12422    1.05552869                      12422    1.05552869

PRINTED ANSWERS
train: 5,491,971 total rows; 5,357,406 distinct full rows; 134,565 extra duplicate copies.
validation: 1,176,851 total rows; 1,170,554 distinct full rows; 6,297 extra duplicate copies.
test: 1,176,851 total rows; 1,170,467 distinct full rows; 6,384 extra duplicate copies.

train vs validation:
  Distinct shared full rows 

### What the overlap results mean

The splits are **not disjoint**. Train shares 57,744 distinct full rows with validation and 57,847 with test; validation shares 12,422 with test. Feature-only shared-pattern counts are the same. In particular, **57,847 test rows (4.91540560% of test)** have features also present in training. The matching-row counts equal the distinct-pattern counts in this dataset, so repeated copies do not increase cross-split coverage here.

Within the files, train contains 134,565 extra duplicate copies, validation 6,297, and test 6,384. These counts remove one original copy of each full row and count only the remaining copies. All 7,845,673 rows were checked using exact stored field strings; no binary label mapping or numeric normalization was applied. The latest completed run took about eight minutes.